# 11 · Generative Evaluation

> **Source notes:** `GenerativeEvaluation.md`

How do you know if a generated image is *good*? This notebook (CPU, no API keys) implements:

- **Toy FID** using a simple CNN feature extractor on MNIST
- **Sample-count bias** — watch FID stabilise as N grows
- **CLIP Score proxy** using sentence-transformers cosine similarity
- **Precision & Recall** via k-NN manifold estimation
- **LPIPS-style perceptual similarity** using VGG features (torchvision)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy",
                "sentence-transformers", "scipy", "-q"], check=True)

import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from torch.utils.data import DataLoader, Subset

torch.manual_seed(42)
print("Ready.")

## 1 · Feature Extractor (proxy for Inception-v3)

In [ ]:
class SmallEncoder(nn.Module):
    """Tiny CNN that acts as our Inception-v3 proxy for MNIST."""
    def __init__(self, feat_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 7x7
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                     # 1x1
            nn.Flatten(),
            nn.Linear(128, feat_dim),
        )
    def forward(self, x):
        return self.net(x)

# Pre-train encoder on MNIST digit classification so features are meaningful
dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))]))
loader  = DataLoader(dataset, batch_size=256, shuffle=True)

encoder = SmallEncoder(feat_dim=128)
classifier_head = nn.Linear(128, 10)
all_params = list(encoder.parameters()) + list(classifier_head.parameters())
opt = torch.optim.Adam(all_params, lr=1e-3)

print("Pre-training encoder on MNIST...")
encoder.train(); classifier_head.train()
for epoch in range(5):
    total = 0
    for imgs, labels in loader:
        feats  = encoder(imgs)
        loss   = F.cross_entropy(classifier_head(feats), labels)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}/5  loss={total/len(loader):.4f}")

encoder.eval()
print("Encoder ready.")

## 2 · FID Implementation

In [ ]:
def get_features(dataloader, enc, n_max=None):
    """Extract feature vectors for all images in dataloader."""
    feats = []
    enc.eval()
    with torch.no_grad():
        for imgs, _ in dataloader:
            feats.append(enc(imgs).numpy())
            if n_max and sum(f.shape[0] for f in feats) >= n_max:
                break
    feats = np.concatenate(feats, axis=0)
    if n_max:
        feats = feats[:n_max]
    return feats

def compute_fid(feats_real, feats_gen):
    """Fréchet distance between two sets of feature vectors."""
    mu_r, sig_r = feats_real.mean(0), np.cov(feats_real, rowvar=False)
    mu_g, sig_g = feats_gen.mean(0),  np.cov(feats_gen,  rowvar=False)
    diff = mu_r - mu_g
    # Matrix square root (can produce small imaginary parts due to float errors)
    cov_sqrt = sqrtm(sig_r @ sig_g)
    if np.iscomplexobj(cov_sqrt):
        cov_sqrt = cov_sqrt.real
    fid = diff @ diff + np.trace(sig_r + sig_g - 2 * cov_sqrt)
    return float(fid)

print("FID functions defined.")

In [ ]:
# Real features: use test set
test_set = torchvision.datasets.MNIST(
    "/tmp/mnist", train=False,
    transform=T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))]))
test_loader = DataLoader(test_set, batch_size=256)
real_feats  = get_features(test_loader, encoder)   # all 10k test images

# "Generated" images: use training images (different split) — should give low FID
train_loader_gen = DataLoader(
    Subset(dataset, range(5000)), batch_size=256, shuffle=False)
gen_feats_good = get_features(train_loader_gen, encoder, n_max=5000)

# "Bad" generated images: pure Gaussian noise — should give high FID
class FakeNoiseDataset(torch.utils.data.Dataset):
    def __len__(self):
        return 5000
    def __getitem__(self, idx):
        return torch.randn(1, 28, 28), 0

noise_loader = DataLoader(FakeNoiseDataset(), batch_size=256)
gen_feats_noise = get_features(noise_loader, encoder)

fid_good  = compute_fid(real_feats[:5000], gen_feats_good)
fid_noise = compute_fid(real_feats[:5000], gen_feats_noise)
print(f"FID (real vs. in-distribution):  {fid_good:.2f}   ← should be LOW")
print(f"FID (real vs. Gaussian noise):   {fid_noise:.2f}  ← should be HIGH")

## 3 · Sample-Count Bias

In [ ]:
sample_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
fids = []
for n in sample_sizes:
    f = compute_fid(real_feats[:n], gen_feats_good[:n])
    fids.append(f)
    print(f"  N={n:5d}  FID={f:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, fids, "o-", color="steelblue", lw=2)
plt.xscale("log")
plt.xlabel("Sample count N")
plt.ylabel("FID")
plt.title("FID bias: estimate stabilises only at large N")
plt.axhline(fids[-1], color="red", ls="--", label=f"N={sample_sizes[-1]} baseline")
plt.legend()
plt.tight_layout()
plt.show()
print("\nKey insight: FID is biased high at small N. Use at least N=5k for comparison.")

## 4 · CLIP Score Proxy

In [ ]:
# We can't run real CLIP easily on CPU without downloading large model weights,
# so we demo the CLIP Score formula with sentence-transformers text embeddings
# and our CNN image encoder as a stand-in.
# In production: use `transformers` CLIPModel or `open_clip`.

from sentence_transformers import SentenceTransformer

text_model  = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast

digit_names = ["zero","one","two","three","four","five","six","seven","eight","nine"]
prompts     = [f"a handwritten digit {d}" for d in digit_names]
text_embs   = text_model.encode(prompts, normalize_embeddings=True)   # (10, 384)

# Projection to align image features (128-dim) with text features (384-dim)
proj = nn.Linear(128, 384, bias=False)
# (In a real CLIP model the two encoders are jointly trained to align)

W = 2.5   # standard CLIP Score scaling constant

samples_loader = DataLoader(test_set, batch_size=100, shuffle=True)
imgs_s, labels_s = next(iter(samples_loader))

with torch.no_grad():
    img_feats = F.normalize(proj(encoder(imgs_s)), dim=-1).numpy()

# For each image, compute cosine sim with its GT text prompt vs. wrong prompts
scores_correct = []
scores_wrong   = []
for i in range(len(labels_s)):
    gt_text  = text_embs[labels_s[i].item()]           # correct label
    bad_text = text_embs[(labels_s[i].item() + 5) % 10]  # wrong label
    scores_correct.append(W * max(0, img_feats[i] @ gt_text))
    scores_wrong.append(  W * max(0, img_feats[i] @ bad_text))

print(f"Avg CLIP-proxy score (correct label): {np.mean(scores_correct):.4f}")
print(f"Avg CLIP-proxy score (wrong label):   {np.mean(scores_wrong):.4f}")
print("\nNote: the gap is modest because our proj layer is random (not jointly trained).")
print("Real CLIP Score values typically range 20–35 for good T2I models.")

## 5 · Precision & Recall via k-NN Manifold

In [ ]:
def knn_manifold(feats, k=5):
    """Return per-point k-NN radius for each feature (defines the manifold boundary)."""
    # Pairwise L2: O(N^2) but fine for N=500
    diff = feats[:, None, :] - feats[None, :, :]   # (N, N, D)
    dists = np.sqrt((diff**2).sum(-1))              # (N, N)
    np.fill_diagonal(dists, np.inf)
    kth_dists = np.sort(dists, axis=1)[:, k-1]     # per-point k-NN radius
    return kth_dists

def precision_recall(real_feats, gen_feats, k=5):
    """Precision = % gen inside real manifold; Recall = % real inside gen manifold."""
    r_r = knn_manifold(real_feats, k)   # shape (N_real,)
    r_g = knn_manifold(gen_feats,  k)   # shape (N_gen,)

    # Precision: gen point g_i is in real manifold if dist(g_i, r_j) <= r_r[j] for ANY j
    dists_gr = np.sqrt(((gen_feats[:, None, :] - real_feats[None, :, :])**2).sum(-1))
    # dists_gr[i, j] = dist(gen[i], real[j]); r_r[j] = radius of real[j]'s k-NN ball
    prec = (dists_gr <= r_r[None, :]).any(axis=1).mean()

    # Recall: real point r_j is covered if dist(r_j, g_i) <= r_g[i] for ANY i
    dists_rg = np.sqrt(((real_feats[:, None, :] - gen_feats[None, :, :])**2).sum(-1))
    rec  = (dists_rg <= r_g[None, :]).any(axis=1).mean()

    return float(prec), float(rec)

N = 300   # keep small for CPU speed
prec_good,  rec_good  = precision_recall(real_feats[:N], gen_feats_good[:N])
prec_noise, rec_noise = precision_recall(real_feats[:N], gen_feats_noise[:N])

print(f"In-distribution:   Precision={prec_good:.2f}  Recall={rec_good:.2f}")
print(f"Gaussian noise:    Precision={prec_noise:.2f}  Recall={rec_noise:.2f}")
print("\nHigh precision + low recall → mode collapse (generates only a few types).")
print("Low precision + high recall → generates diverse but low-quality images.")

## 5b · LPIPS-style Perceptual Similarity

In [ ]:
import torchvision.models as models

class LPIPSProxy(nn.Module):
    """
    Lightweight perceptual similarity using VGG-11 feature layers.
    Real LPIPS (Zhang et al. 2018) uses AlexNet/VGG with learned channel weights.
    This proxy uses the raw feature activations (cosine similarity) to show the mechanism.
    """
    def __init__(self):
        super().__init__()
        vgg = models.vgg11(weights=None)   # random init: shows mechanism, no download needed
        # Use features up through block2_conv2 (shallow spatial features)
        self.layers = nn.Sequential(*list(vgg.features[:8]))
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        # MNIST is 1-channel; VGG expects 3-channel
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(32, 32), mode="bilinear", align_corners=False)
        return self.layers(x).flatten(1)   # (B, F)

lpips_proxy = LPIPSProxy()

# Load a small batch of test images
imgs_test, labels_test = next(iter(DataLoader(test_set, batch_size=64, shuffle=True)))

with torch.no_grad():
    feats_test = lpips_proxy(imgs_test)   # (64, F)

# Compare same-class pairs vs. different-class pairs
same_sims, diff_sims = [], []
indices = list(range(0, 60, 2))
for i in indices:
    sim = F.cosine_similarity(feats_test[i:i+1], feats_test[i+1:i+2]).item()
    if labels_test[i] == labels_test[i + 1]:
        same_sims.append(sim)
    else:
        diff_sims.append(sim)

print(f"Perceptual similarity — same class: {np.mean(same_sims):.4f}  ← should be HIGHER")
print(f"Perceptual similarity — diff class: {np.mean(diff_sims):.4f}  ← should be LOWER")
print()
print("LPIPS in practice: uses learned channel weights on AlexNet/VGG layers.")
print("Lower LPIPS = more perceptually similar (distance metric, not similarity).")
print("Here we report cosine similarity for readability: higher = more similar.")

## 6 · FID vs. Guidance Scale (PixelSmith Evaluation)

In [ ]:
# Simulate the U-shaped FID vs. guidance scale curve:
#   Too-low w  → samples are diverse but off-distribution (mean shifts away)
#   Sweet spot → best match to the real distribution
#   Too-high w → mode-collapsed: near-zero variance, also a poor match

real_mean = real_feats[:2000].mean(0)
real_std  = real_feats[:2000].std(0)

def sim_generated(guidance_w, n=2000, optimal_w=7.0):
    """
    Simulate the effect of guidance scale on the generated feature distribution.
    - w < optimal: guidance too weak → samples drift off the real manifold (mean shift)
    - w = optimal: best distribution match → lowest FID
    - w > optimal: guidance too strong → mode collapse (variance crushed → high FID)
    """
    delta = guidance_w - optimal_w

    # Below optimal: mean shifts away from real_mean (incoherent generation)
    mean_shift_scale = max(0.0, -delta) * 0.15
    mean_shift = mean_shift_scale * real_std

    # Above optimal: variance crushed (mode collapse)
    variance_scale = max(0.05, 1.0 - max(0.0, delta) * 0.12)

    feats = (real_mean + mean_shift) + variance_scale * np.random.randn(n, 128) * real_std
    return feats

guidance_scales = [0.0, 1.0, 2.0, 4.0, 7.0, 10.0, 15.0]
fids_guidance   = [compute_fid(real_feats[:2000], sim_generated(w)) for w in guidance_scales]

plt.figure(figsize=(8, 4))
plt.plot(guidance_scales, fids_guidance, "s-", color="darkorange", lw=2)
plt.xlabel("Guidance scale w")
plt.ylabel("FID (simulated)")
plt.title("FID vs. Guidance Scale — U-shaped; sweet spot near w=7")
best_w = guidance_scales[int(np.argmin(fids_guidance))]
plt.axvline(best_w, color="green", ls="--", label=f"Best w={best_w:.0f}")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Lowest FID at guidance scale w={best_w:.0f} (expected: ~7)")

## 7 · Metric Comparison Table

In [ ]:
print(f"{'Metric':<20} {'Direction':<10} {'Ref needed':<12} {'Sample scale':<14} {'Use case'}")
print("-" * 80)
rows = [
    ("FID",           "↓ lower",  "Yes (real)",  "≥5k–50k",    "Unconditional / T2I realism"),
    ("IS",            "↑ higher", "No",          "≥5k",        "Legacy; rarely used alone"),
    ("CLIP Score",    "↑ higher", "No",          "Per-sample", "Text-image alignment"),
    ("LPIPS",         "↓ lower",  "Yes (ref img)","Per-sample","img2img / inpainting"),
    ("Precision",     "↑ higher", "Yes (real)",  "≥2k",        "Fidelity (no mode collapse)"),
    ("Recall",        "↑ higher", "Yes (real)",  "≥2k",        "Diversity (distribution cover)"),
    ("HPSv2/PickScore","↑ higher","No",          "Per-sample", "Human preference proxy"),
    ("FVD",           "↓ lower",  "Yes (real)",  "≥2k clips", "Video generation quality"),
]
for r in rows:
    print(f"{r[0]:<20} {r[1]:<10} {r[2]:<12} {r[3]:<14} {r[4]}")

## 8 · Summary

| Experiment | Takeaway |
|-----------|----------|
| FID sanity check | Real vs. in-dist = low FID; real vs. noise = high FID ✓ |
| Sample-count bias | FID inflated at N=50–100; stabilises around N=5k |
| CLIP Score proxy | Correct-label score > wrong-label score |
| LPIPS proxy | Same-class pairs score higher than cross-class pairs ✓ |
| Precision/Recall | Noise scores lower on both axes |
| Guidance FID curve | U-shape — both too low and too high w raise FID |

**Next:** [LocalDiffusionLab.md](../LocalDiffusionLab/LocalDiffusionLab.md) — Capstone unifying everything.